# EdgeVerify — Reproduction & Verification Notebook

This notebook reproduces the EdgeVerify experiments in Google Colab and extends them to the **real CIFAR-10 benchmark** (which the original run could not download).

**How to use:** `Runtime -> Run all`. CPU is sufficient (a few minutes); a GPU runtime is faster but not required.

It has three parts:
1. Recreate the exact repo files (`model.py`, `dataset.py`, `evaluate.py`).
2. Re-run the **structured** and **digits** experiments — these use a fixed seed, so your numbers should match the reference table below.
3. Run a new **CIFAR-10** experiment (real natural images) as the benchmark test.

> Note: this notebook is self-contained; it does not need the GitHub repo. If you prefer, you can instead `!git clone` your repo once you have pushed these files to it.


## 0. Install dependencies


In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib


## 1. Recreate the repo files
The next three cells write the *exact* source used in the thesis to disk, so the run is identical to the reported one.


In [ ]:
%%writefile model.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_stable_local_loss(projected_student, teacher_target, layer_output,
                              variance_threshold=1.0, alpha=1.0, beta=0.01,
                              return_components=False):
    """
    Computes regularized local distillation loss using VicReg-style constraints
    to prevent dimensional collapse.

    The default coefficients (alpha=1.0, beta=0.01) reproduce the original
    formulation exactly. They are exposed as arguments so that the
    unregularized baseline (alpha=beta=0) required for the collapse ablation
    can share this single code path. Set return_components=True to obtain the
    per-term breakdown for logging.
    """
    # 1. Base Distillation Loss (MSE against target)
    distill_loss = F.mse_loss(projected_student, teacher_target)

    # 2. Variance Constraint (Hinge loss on batch standard deviation)
    std_student = torch.sqrt(layer_output.var(dim=0) + 1e-4)
    variance_loss = torch.mean(F.relu(variance_threshold - std_student))

    # 3. Covariance Regularization (Feature decorrelation)
    centered_student = projected_student - projected_student.mean(dim=0)
    batch_size = projected_student.size(0)
    cov_matrix = (centered_student.T @ centered_student) / (batch_size - 1)
    diag_mask = torch.eye(cov_matrix.size(0), device=projected_student.device)
    covariance_loss = (cov_matrix * (1 - diag_mask)).pow(2).sum() / cov_matrix.size(0)

    total = distill_loss + alpha * variance_loss + beta * covariance_loss
    if return_components:
        return total, {
            "distill": distill_loss.item(),
            "var": variance_loss.item(),
            "cov": covariance_loss.item(),
            "total": total.item(),
        }
    return total

class VisionJEPA(nn.Module):
    def __init__(self, img_channels=3, latent_dim=256, ema_decay=0.999):
        super().__init__()
        self.latent_dim = latent_dim
        self.ema_decay = ema_decay

        # Context Encoder (f_theta)
        self.context_encoder = nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, latent_dim, kernel_size=3, stride=2, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        # Target Encoder (f_theta_bar) - Updated via EMA
        self.target_encoder = copy.deepcopy(self.context_encoder)
        for p in self.target_encoder.parameters():
            p.requires_grad = False

        # Action/Bounding-Box Encoder
        self.action_encoder = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )

        # Latent Predictor Block (p_psi)
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim * 2, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim)
        )

    @torch.no_grad()
    def update_target_encoder(self):
        for p_ctx, p_tgt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            p_tgt.data.mul_(self.ema_decay).add_(p_ctx.data, alpha=1.0 - self.ema_decay)

    def forward(self, partial_images, full_images, spatial_actions):
        s_t = self.context_encoder(partial_images)
        with torch.no_grad():
            s_target = self.target_encoder(full_images)
        a_t = self.action_encoder(spatial_actions)

        combined_latent = torch.cat([s_t, a_t], dim=-1)
        s_predicted = self.predictor(combined_latent)

        return s_predicted, s_target, s_t


In [ ]:
%%writefile dataset.py
"""
Dataset generators and masking loaders for EdgeVerify.

Two sources are provided:

  * ``StructuredJEPADataset`` -- procedurally generated 64x64 RGB images with
    genuine spatial structure (background gradients plus randomly placed
    rectangles and circles). Unlike i.i.d. uniform noise, these images contain
    predictable local structure, so a context encoder can learn a non-trivial
    representation and the collapse ablation is meaningful.

  * ``DigitsJEPADataset`` -- the scikit-learn handwritten-digits dataset
    (real 8x8 images, 1797 samples), upsampled to 64x64 and replicated to
    three channels. This is real (non-synthetic) data available offline, used
    as a sanity check.

Each sample is a triple ``(partial_image, full_image, spatial_action)`` matching
the signature of ``VisionJEPA.forward``. The partial (context) view is produced
by zeroing a rectangular region of the full image; the spatial action is the
normalized bounding box ``[x, y, w, h]`` of that masked region, so the predictor
is conditioned on the location it must reconstruct in latent space.
"""

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset


def _apply_mask(img, rng, min_frac=0.35, max_frac=0.6):
    """Zero a random rectangular region; return (partial, action[x,y,w,h] normalized)."""
    _, H, W = img.shape
    w = int(rng.uniform(min_frac, max_frac) * W)
    h = int(rng.uniform(min_frac, max_frac) * H)
    x = int(rng.integers(0, W - w + 1))
    y = int(rng.integers(0, H - h + 1))
    partial = img.clone()
    partial[:, y:y + h, x:x + w] = 0.0
    action = torch.tensor([x / W, y / H, w / W, h / H], dtype=torch.float32)
    return partial, action


def _make_structured_image(rng, size=64):
    """Background gradient + a few random rectangles and circles, values in [0,1]."""
    img = np.zeros((3, size, size), dtype=np.float32)
    # Background gradient between two random colours.
    c0 = rng.uniform(0, 1, size=3)
    c1 = rng.uniform(0, 1, size=3)
    if rng.random() < 0.5:
        ramp = np.linspace(0, 1, size)[None, :]        # horizontal
    else:
        ramp = np.linspace(0, 1, size)[:, None]        # vertical
    for c in range(3):
        img[c] = c0[c] + (c1[c] - c0[c]) * ramp
    # Random rectangles.
    yy, xx = np.mgrid[0:size, 0:size]
    for _ in range(int(rng.integers(1, 4))):
        rw, rh = rng.integers(8, 28), rng.integers(8, 28)
        rx, ry = rng.integers(0, size - rw), rng.integers(0, size - rh)
        col = rng.uniform(0, 1, size=3)
        img[:, ry:ry + rh, rx:rx + rw] = col[:, None, None]
    # Random filled circles.
    for _ in range(int(rng.integers(1, 4))):
        cx, cy = rng.integers(10, size - 10), rng.integers(10, size - 10)
        r = rng.integers(5, 14)
        col = rng.uniform(0, 1, size=3)
        disk = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
        for c in range(3):
            img[c][disk] = col[c]
    return torch.from_numpy(np.clip(img, 0.0, 1.0))


class StructuredJEPADataset(Dataset):
    def __init__(self, n_samples=1024, size=64, seed=0):
        rng = np.random.default_rng(seed)
        self.samples = []
        for _ in range(n_samples):
            full = _make_structured_image(rng, size)
            partial, action = _apply_mask(full, rng)
            self.samples.append((partial, full, action))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class DigitsJEPADataset(Dataset):
    def __init__(self, size=64, seed=0):
        from sklearn.datasets import load_digits
        rng = np.random.default_rng(seed)
        digits = load_digits()
        imgs = digits.images.astype(np.float32) / 16.0          # (1797, 8, 8), scaled to [0,1]
        t = torch.from_numpy(imgs).unsqueeze(1)                   # (N, 1, 8, 8)
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        t = t.repeat(1, 3, 1, 1)                                  # (N, 3, size, size)
        self.samples = []
        for i in range(t.size(0)):
            full = t[i].contiguous()
            partial, action = _apply_mask(full, rng)
            self.samples.append((partial, full, action))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def get_dataset(name, **kwargs):
    name = name.lower()
    if name in ("structured", "shapes"):
        return StructuredJEPADataset(**kwargs)
    if name in ("digits", "sklearn"):
        return DigitsJEPADataset(**kwargs)
    raise ValueError(f"Unknown dataset: {name!r}")


In [ ]:
%%writefile evaluate.py
"""
Evaluation protocol for EdgeVerify.

Runs the collapse ablation (full VICReg-style objective vs. an unregularized
alpha=beta=0 baseline) on each available dataset, and measures:

  RQ1 (collapse) : mean per-dimension embedding std (sigma_bar) and the
                   effective rank (participation ratio) of the context-embedding
                   covariance. Higher is healthier; collapse drives both to ~0/1.
  RQ2 (efficiency): parameter count, parameter memory, single-sample inference
                   latency (mean +/- std), and peak process RSS during training.
  RQ3 (convergence): per-epoch loss components.

Outputs a results JSON and PNG figures for inclusion in the thesis.
"""

import json
import os
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

from model import VisionJEPA, compute_stable_local_loss
from dataset import get_dataset


def _rss_mb():
    """Resident set size of this process in MB, read from /proc/self/status."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return float(line.split()[1]) / 1024.0  # kB -> MB
    except FileNotFoundError:
        pass
    return float("nan")


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


def train_model(dataset, epochs, alpha, beta, batch_size=64, lr=1e-3, wd=1e-4, seed=0):
    set_seed(seed)
    device = torch.device("cpu")
    model = VisionJEPA().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    history = []
    peak_rss = _rss_mb()
    for epoch in range(epochs):
        agg = {"distill": 0.0, "var": 0.0, "cov": 0.0, "total": 0.0}
        n = 0
        for partial, full, action in loader:
            partial, full, action = partial.to(device), full.to(device), action.to(device)
            optimizer.zero_grad()
            s_pred, s_tgt, s_t = model(partial, full, action)
            loss, comps = compute_stable_local_loss(
                s_pred, s_tgt.detach(), s_t,
                variance_threshold=1.0, alpha=alpha, beta=beta, return_components=True)
            loss.backward()
            optimizer.step()
            model.update_target_encoder()
            for k in agg:
                agg[k] += comps[k]
            n += 1
            peak_rss = max(peak_rss, _rss_mb())
        history.append({k: agg[k] / max(n, 1) for k in agg})
    return model, history, peak_rss


@torch.no_grad()
def collapse_metrics(model, dataset, max_samples=512):
    model.eval()
    device = torch.device("cpu")
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    embs = []
    seen = 0
    for partial, full, action in loader:
        s_t = model.context_encoder(partial.to(device))
        embs.append(s_t)
        seen += s_t.size(0)
        if seen >= max_samples:
            break
    E = torch.cat(embs, dim=0)[:max_samples]                 # (M, d)
    sigma_bar = torch.sqrt(E.var(dim=0) + 1e-8).mean().item()
    Ec = E - E.mean(dim=0, keepdim=True)
    cov = (Ec.T @ Ec) / (E.size(0) - 1)
    eig = torch.linalg.eigvalsh(cov).clamp(min=0)
    s1, s2 = eig.sum().item(), (eig ** 2).sum().item()
    eff_rank = (s1 ** 2) / s2 if s2 > 0 else 0.0             # participation ratio
    return {"sigma_bar": sigma_bar, "effective_rank": eff_rank, "latent_dim": E.size(1)}


@torch.no_grad()
def efficiency_metrics(model, n_runs=50, warmup=10):
    model.eval()
    device = torch.device("cpu")
    n_params = sum(p.numel() for p in model.parameters())
    param_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)
    partial = torch.rand(1, 3, 64, 64)
    full = torch.rand(1, 3, 64, 64)
    action = torch.rand(1, 4)
    for _ in range(warmup):
        model(partial, full, action)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model(partial, full, action)
        times.append((time.perf_counter() - t0) * 1000.0)     # ms
    times = np.array(times)
    return {
        "params": int(n_params),
        "param_memory_mb": round(param_mb, 3),
        "latency_ms_mean": round(float(times.mean()), 3),
        "latency_ms_std": round(float(times.std()), 3),
    }


def run_dataset(name, epochs, ds_kwargs, out_img_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    dataset = get_dataset(name, **ds_kwargs)
    results = {}
    curves = {}
    for tag, (alpha, beta) in {"full": (1.0, 0.01), "unregularized": (0.0, 0.0)}.items():
        model, history, peak_rss = train_model(dataset, epochs, alpha, beta, seed=0)
        col = collapse_metrics(model, dataset)
        eff = efficiency_metrics(model)
        results[tag] = {
            "alpha": alpha, "beta": beta,
            "final_loss": history[-1],
            "collapse": col,
            "efficiency": eff,
            "peak_rss_mb": round(peak_rss, 1),
        }
        curves[tag] = history

    # Loss-curve figure (total loss per epoch, both configs).
    plt.figure(figsize=(6, 4))
    for tag in curves:
        plt.plot([h["total"] for h in curves[tag]], label=tag, linewidth=2)
    plt.xlabel("Epoch"); plt.ylabel(r"$\mathcal{L}_{total}$")
    plt.title(f"Convergence on {name} data"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    loss_path = os.path.join(out_img_dir, f"loss_curves_{name}.png")
    plt.savefig(loss_path, dpi=150); plt.close()

    return results, loss_path


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    img_dir = os.path.abspath(os.path.join(here, "..", "latex_report", "images"))
    res_dir = os.path.abspath(os.path.join(here, "..", "results"))
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(res_dir, exist_ok=True)

    all_results = {}
    plans = [
        ("structured", 40, {"n_samples": 768, "seed": 0}),
        ("digits", 25, {"seed": 0}),
    ]
    for name, epochs, kw in plans:
        print(f"\n=== Running {name} (epochs={epochs}) ===", flush=True)
        res, _ = run_dataset(name, epochs, kw, img_dir)
        all_results[name] = res
        for tag, r in res.items():
            c, e = r["collapse"], r["efficiency"]
            print(f"  [{tag:13s}] total={r['final_loss']['total']:.4f} "
                  f"sigma_bar={c['sigma_bar']:.4f} eff_rank={c['effective_rank']:.2f}/"
                  f"{c['latent_dim']} lat={e['latency_ms_mean']:.2f}ms "
                  f"params={e['params']:,} peakRSS={r['peak_rss_mb']:.0f}MB", flush=True)

    # Collapse comparison bar chart (structured dataset).
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for name in all_results:
        r = all_results[name]
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
        tags = ["unregularized", "full"]
        ax[0].bar(tags, [r[t]["collapse"]["sigma_bar"] for t in tags],
                  color=["#c0504d", "#4f81bd"])
        ax[0].set_title(r"Mean embedding std $\bar{\sigma}$")
        ax[1].bar(tags, [r[t]["collapse"]["effective_rank"] for t in tags],
                  color=["#c0504d", "#4f81bd"])
        ax[1].set_title("Effective rank")
        for a in ax:
            a.grid(alpha=0.3, axis="y")
        fig.suptitle(f"Collapse indicators ({name} data)")
        fig.tight_layout()
        fig.savefig(os.path.join(img_dir, f"collapse_bars_{name}.png"), dpi=150)
        plt.close(fig)

    with open(os.path.join(res_dir, "results.json"), "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved results.json and figures to {res_dir} and {img_dir}", flush=True)


if __name__ == "__main__":
    main()


## 2. Reproduce structured + digits

Running `evaluate.py` trains the full objective and the unregularized (alpha=beta=0) baseline on both datasets and prints collapse + efficiency metrics.

### Reference numbers from the original cloud run (CPU)

| Data | Model | L_total | sigma_bar | Eff. rank |
|---|---|---|---|---|
| Structured | Unregularized | 0.0021 | 0.245 | 6.46 |
| Structured | EdgeVerify (full) | 0.190 | 1.549 | 1.64 |
| Digits | Unregularized | 0.0006 | 0.080 | 3.57 |
| Digits | EdgeVerify (full) | 0.257 | 1.772 | 1.03 |

Params: 745,536 | Param memory: 2.84 MB. Your `sigma_bar` / effective-rank pattern should match closely; exact loss values can vary slightly with library versions and thread count, but the **qualitative finding** (variance term raises sigma_bar; effective rank stays low for the full model) should reproduce.


In [ ]:
!python evaluate.py


## 3. Real benchmark: CIFAR-10

This is the test the original environment could not run (the dataset host was network-blocked). Here we download CIFAR-10, resize to 64x64, apply the same random masking, and run the same full-vs-unregularized comparison.

We subsample for speed; increase `N_SAMPLES` / `EPOCHS` (or switch to a GPU runtime) for a fuller run.


In [ ]:
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision

from dataset import _apply_mask
from evaluate import train_model, collapse_metrics, efficiency_metrics

class CIFARJEPADataset(Dataset):
    def __init__(self, n_samples=2000, size=64, seed=0):
        rng = np.random.default_rng(seed)
        tv = torchvision.datasets.CIFAR10(root="./cifar", train=True, download=True)
        data = torch.from_numpy(tv.data[:n_samples]).float().permute(0, 3, 1, 2) / 255.0  # (N,3,32,32)
        data = F.interpolate(data, size=(size, size), mode="bilinear", align_corners=False)
        self.samples = []
        for i in range(data.size(0)):
            full = data[i].contiguous()
            partial, action = _apply_mask(full, rng)
            self.samples.append((partial, full, action))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

N_SAMPLES, EPOCHS = 2000, 15
ds = CIFARJEPADataset(n_samples=N_SAMPLES, seed=0)
print(f"CIFAR-10 subset: {len(ds)} images, {EPOCHS} epochs\n")

results = {}
for tag, (alpha, beta) in {"full": (1.0, 0.01), "unregularized": (0.0, 0.0)}.items():
    model, history, peak = train_model(ds, EPOCHS, alpha, beta, seed=0)
    col = collapse_metrics(model, ds)
    eff = efficiency_metrics(model)
    results[tag] = (history, col, eff)
    print(f"[{tag:13s}] total={history[-1]['total']:.4f} "
          f"sigma_bar={col['sigma_bar']:.4f} eff_rank={col['effective_rank']:.2f}/{col['latent_dim']} "
          f"lat={eff['latency_ms_mean']:.2f}ms params={eff['params']:,}")


### Plot the CIFAR-10 results


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for tag in results:
    ax[0].plot([h["total"] for h in results[tag][0]], label=tag, linewidth=2)
ax[0].set_title("CIFAR-10 convergence"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("L_total")
ax[0].legend(); ax[0].grid(alpha=0.3)

tags = ["unregularized", "full"]
ax[1].bar(tags, [results[t][1]["sigma_bar"] for t in tags], color=["#c0504d", "#4f81bd"])
ax[1].set_title(r"Mean embedding std $\bar{\sigma}$"); ax[1].grid(alpha=0.3, axis="y")
ax[2].bar(tags, [results[t][1]["effective_rank"] for t in tags], color=["#c0504d", "#4f81bd"])
ax[2].set_title("Effective rank"); ax[2].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print("\\nInterpretation: compare against the thesis finding — does the variance term keep sigma_bar high on")
print("real images, and does effective rank stay low at beta=0.01? Try beta=0.1 or 1.0 above to test the ablation.")
